# CUDA GEMV

what is GEMV (General Matrix-Vector Multiplication) is memory problem, like how much meory you will move between on vs off-chip for the matrix operations 

in this article we see how much can we write custom kernels and optimize then native Nvidia cublas , cuda and Pytorch


keep in mind that we perform Wx + b


Machine specs 
* GPU name:           NVIDIA GeForce RTX 4050 Laptop GPU
* CUDA version:       13.0
* PyTorch version:    2.12.1+cu130
* Compute capability: 8.9
* Total GPU memory:   5.64 GiB 

## 1. Setup 

In [89]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from torch.utils.cpp_extension import load_inline

assert torch.cuda.is_available(), (
    "CUDA GPU is required. Select a CUDA-enabled Python environment and verify "
    "that the NVIDIA driver is visible before running this notebook."
)

DEVICE_INDEX = torch.cuda.current_device()
DEVICE = torch.device(f"cuda:{DEVICE_INDEX}")
GPU = torch.cuda.get_device_properties(DEVICE_INDEX)

# Keep the FP32 benchmark as FP32 math instead of silently using TF32 tensor cores.
torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("highest")

torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
pd.set_option("display.precision", 4)

print(f"GPU name:           {GPU.name}")
print(f"CUDA version:       {torch.version.cuda}")
print(f"PyTorch version:    {torch.__version__}")
print(f"Compute capability: {GPU.major}.{GPU.minor}")
print(f"Total GPU memory:   {GPU.total_memory / 2**30:.2f} GiB")
print(f"TF32 matmul:        {torch.backends.cuda.matmul.allow_tf32}")


GPU name:           NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version:       13.0
PyTorch version:    2.12.1+cu130
Compute capability: 8.9
Total GPU memory:   5.64 GiB
TF32 matmul:        False


## 2. GEMV problem definition

GEMV is mostly a memory movement problem, not a math problem.

we read the W from the HBM



In [90]:
M = 10096
N = 10096
PEAK_BANDWIDTH_GBPS = 1008.0  # Change this for your GPU before interpreting peak %.

BENCHMARK_CASES = [
    {
        "precision": "FP16",
        "dtype": torch.float16,
        "rtol": 1e-2,
        "atol": 1e-1,
    },
    {
        "precision": "FP32",
        "dtype": torch.float32,
        "rtol": 1e-4,
        "atol": 1e-3,
    },
]

PROBLEMS = {}
for case in BENCHMARK_CASES:
    precision = case["precision"]
    dtype = case["dtype"]
    W = torch.randn((M, N), device=DEVICE, dtype=dtype).contiguous()
    x = torch.randn((N,), device=DEVICE, dtype=dtype).contiguous()

    assert W.shape == (M, N) and x.shape == (N,)
    assert W.is_contiguous() and x.is_contiguous()

    weight_bytes = M * N * W.element_size()
    x_bytes = N * x.element_size()
    y_bytes = M * torch.empty((), dtype=dtype).element_size()
    logical_memory_bytes = weight_bytes + x_bytes + y_bytes

    PROBLEMS[precision] = {
        **case,
        "W": W,
        "x": x,
        "weight_bytes": weight_bytes,
        "x_bytes": x_bytes,
        "y_bytes": y_bytes,
        "logical_memory_bytes": logical_memory_bytes,
    }

    print(f"{precision}")
    print(f"  W: {tuple(W.shape)}, {W.dtype}, {weight_bytes / 1e6:.3f} MB")
    print(f"  x: {tuple(x.shape)}, {x.dtype}, {x_bytes / 1e3:.3f} KB")
    print(f"  y: ({M},), {dtype}, {y_bytes / 1e3:.3f} KB")
    print(f"  Logical traffic per GEMV: {logical_memory_bytes / 1e6:.3f} MB")


FP16
  W: (10096, 10096), torch.float16, 203.858 MB
  x: (10096,), torch.float16, 20.192 KB
  y: (10096,), torch.float16, 20.192 KB
  Logical traffic per GEMV: 203.899 MB
FP32
  W: (10096, 10096), torch.float32, 407.717 MB
  x: (10096,), torch.float32, 40.384 KB
  y: (10096,), torch.float32, 40.384 KB
  Logical traffic per GEMV: 407.798 MB


## 3. Inline CUDA extension through torch

In [91]:
GPU.major , GPU.minor 

(8, 9)

In [92]:
from pathlib import Path

def resolve_source_file(filename):
    candidates = (
        Path.cwd() / filename,
        Path.cwd() / "kernels" / filename,
    )
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    searched = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")

def cublas_link_flags():
    # Prefer the cuBLAS major bundled with PyTorch. This matters when the
    # system CUDA toolkit and the PyTorch CUDA build have different majors.
    cuda_major = torch.version.cuda.split(".", 1)[0]
    site_packages = Path(torch.__file__).resolve().parent.parent
    candidates = (
        site_packages / "nvidia" / f"cu{cuda_major}" / "lib"
        / f"libcublas.so.{cuda_major}",
        site_packages / "nvidia" / "cublas" / "lib"
        / f"libcublas.so.{cuda_major}",
    )
    for library in candidates:
        if library.is_file():
            return [str(library), f"-Wl,-rpath,{library.parent}"]
    return ["-lcublas"]

CPP_SOURCE_PATH = resolve_source_file("gemv_bindings.cpp")
CUDA_SOURCE_PATH = resolve_source_file("gemv_kernels.cu")
cpp_source = CPP_SOURCE_PATH.read_text(encoding="utf-8")
cuda_source = CUDA_SOURCE_PATH.read_text(encoding="utf-8")
CUBLAS_LINK_FLAGS = cublas_link_flags()

print(f"C++ bindings: {CPP_SOURCE_PATH}")
print(f"CUDA kernels: {CUDA_SOURCE_PATH}")
print(f"cuBLAS link flags: {CUBLAS_LINK_FLAGS}")

C++ bindings: /data/inference-book/kernels/GEMV/gemv_bindings.cpp
CUDA kernels: /data/inference-book/kernels/GEMV/gemv_kernels.cu
cuBLAS link flags: ['/data/inference-book/.venv/lib/python3.12/site-packages/nvidia/cu13/lib/libcublas.so.13', '-Wl,-rpath,/data/inference-book/.venv/lib/python3.12/site-packages/nvidia/cu13/lib']


In [93]:
# Compile the external native sources through PyTorch's inline extension cache.
extension_name = f"gemv_cuda_ext_v3_same_dtype_sm{GPU.major}{GPU.minor}"
gemv_ext = load_inline(
    name=extension_name,
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=None,
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3", "-lineinfo"],
    extra_ldflags=CUBLAS_LINK_FLAGS,
    with_cuda=True,
    verbose=True,
)

print(f"Loaded inline extension: {extension_name}")


Loaded inline extension: gemv_cuda_ext_v3_same_dtype_sm89


## 4. PyTorch baseline 

In [94]:
RUNNERS = {precision: {} for precision in PROBLEMS}

for precision, problem in PROBLEMS.items():
    W = problem["W"]
    x = problem["x"]
    y_torch = torch.empty((M,), device=DEVICE, dtype=problem["dtype"])

    def run_pytorch(W=W, x=x, y_torch=y_torch):
        torch.matmul(W, x, out=y_torch)
        return y_torch

    RUNNERS[precision]["PyTorch torch.matmul"] = run_pytorch
    result = run_pytorch()
    torch.cuda.synchronize()
    print(f"{precision} PyTorch result dtype: {result.dtype}")


FP16 PyTorch result dtype: torch.float16
FP32 PyTorch result dtype: torch.float32


## 5. Naive approach

In [95]:
for precision, problem in PROBLEMS.items():
    W = problem["W"]
    x = problem["x"]
    y_naive = torch.empty((M,), device=DEVICE, dtype=problem["dtype"])

    def run_naive_cuda(W=W, x=x, y_naive=y_naive):
        gemv_ext.naive_gemv_out(W, x, y_naive)
        return y_naive

    RUNNERS[precision]["Naive CUDA"] = run_naive_cuda
    run_naive_cuda()
    torch.cuda.synchronize()
    print(f"{precision} Naive CUDA result dtype: {y_naive.dtype}")


FP16 Naive CUDA result dtype: torch.float16
FP32 Naive CUDA result dtype: torch.float32


## 6. Optimized CUDA GEMV

We compare four custom optimized variants:

- `Optimized CUDA`: one 256-thread block computes one output row.
- `Optimized CUDA v1`: one warp computes one output row, so there is no shared-memory row reduction.
- `Optimized CUDA v2`: one warp computes one row and the block stages vector tiles in shared memory.
- `Optimized CUDA v3`: for FP16, the block stages the full vector in shared memory when it fits; FP32 falls back to v1.

All variants use same-dtype inputs/outputs for each benchmark case and accumulate partial sums in FP32.


In [96]:
OPTIMIZED_VARIANTS = [
    ("Optimized CUDA", gemv_ext.optimized_gemv_out),
    ("Optimized CUDA v1", gemv_ext.optimized_gemv_v1_out),
    ("Optimized CUDA v2", gemv_ext.optimized_gemv_v2_out),
    ("Optimized CUDA v3", gemv_ext.optimized_gemv_v3_out),
]

for precision, problem in PROBLEMS.items():
    W = problem["W"]
    x = problem["x"]

    for method_name, kernel in OPTIMIZED_VARIANTS:
        y_optimized = torch.empty((M,), device=DEVICE, dtype=problem["dtype"])

        def run_optimized_cuda(W=W, x=x, y_optimized=y_optimized, kernel=kernel):
            kernel(W, x, y_optimized)
            return y_optimized

        RUNNERS[precision][method_name] = run_optimized_cuda
        run_optimized_cuda()
        torch.cuda.synchronize()
        print(f"{precision} {method_name} result dtype: {y_optimized.dtype}")


FP16 Optimized CUDA result dtype: torch.float16
FP16 Optimized CUDA v1 result dtype: torch.float16
FP16 Optimized CUDA v2 result dtype: torch.float16
FP16 Optimized CUDA v3 result dtype: torch.float16
FP32 Optimized CUDA result dtype: torch.float32
FP32 Optimized CUDA v1 result dtype: torch.float32
FP32 Optimized CUDA v2 result dtype: torch.float32
FP32 Optimized CUDA v3 result dtype: torch.float32


## 7. Direct cuBLAS GEMV

This path calls `cublasGemmEx` directly from the CUDA extension. We express GEMV as a one-column GEMM:

[M, N] @ [N, 1] -> [M, 1]

This gives us a direct vendor-library baseline without going through `torch.matmul`.

In [97]:
for precision, problem in PROBLEMS.items():
    W = problem["W"]
    x = problem["x"]
    y_cublas = torch.empty((M,), device=DEVICE, dtype=problem["dtype"])

    def run_cublas_direct(W=W, x=x, y_cublas=y_cublas):
        gemv_ext.cublas_gemv_out(W, x, y_cublas)
        return y_cublas

    RUNNERS[precision]["cuBLAS direct"] = run_cublas_direct
    run_cublas_direct()
    torch.cuda.synchronize()
    print(f"{precision} Direct cuBLAS result dtype: {y_cublas.dtype}")


FP16 Direct cuBLAS result dtype: torch.float16
FP32 Direct cuBLAS result dtype: torch.float32


## 8. Correctness validation

Validation uses a high-precision PyTorch FP32 reference and then casts that reference to the benchmark dtype. Assertions are deliberately visible: the notebook stops if any method exceeds the dtype-specific tolerance.


In [98]:
def error_metrics(actual, expected):
    actual_f = actual.float()
    expected_f = expected.float()
    abs_error = (actual_f - expected_f).abs()
    rel_error = abs_error / expected_f.abs().clamp_min(1e-6)
    return {
        "max_abs_error": abs_error.max().item(),
        "mean_abs_error": abs_error.mean().item(),
        "max_rel_error": rel_error.max().item(),
    }

references = {}
correctness = {}

for precision, problem in PROBLEMS.items():
    W = problem["W"]
    x = problem["x"]
    dtype = problem["dtype"]
    reference_fp32 = torch.matmul(W.float(), x.float())
    reference = reference_fp32.to(dtype=dtype)
    references[precision] = reference
    torch.cuda.synchronize()

    for method, fn in RUNNERS[precision].items():
        output = fn()
        torch.cuda.synchronize()
        correctness[(precision, method)] = error_metrics(output, reference)
        torch.testing.assert_close(
            output,
            reference,
            rtol=problem["rtol"],
            atol=problem["atol"],
        )

correctness_df = pd.DataFrame(
    [
        {"precision": precision, "method": method, **metrics}
        for (precision, method), metrics in correctness.items()
    ]
)
display(correctness_df)


,precision,method,max_abs_error,mean_abs_error,max_rel_error
0,FP16,PyTorch torch.matmul,0.0000,0.0000e+00,0.0000
1,FP16,Naive CUDA,0.1250,1.8976e-04,0.0127
2,FP16,Optimized CUDA,0.0625,1.1013e-05,0.0045
3,FP16,Optimized CUDA v1,0.0625,1.3278e-05,0.0054
4,FP16,Optimized CUDA v2,0.0625,1.6645e-05,0.0040
5,FP16,Optimized CUDA v3,0.0625,1.3278e-05,0.0054
6,FP16,cuBLAS direct,0.0000,0.0000e+00,0.0000
7,FP32,PyTorch torch.matmul,0.0000,0.0000e+00,0.0000
8,FP32,Naive CUDA,0.0012,1.2932e-04,0.0154
9,FP32,Optimized CUDA,0.0001,2.7838e-05,0.0085


## 10. Metric calculations

In [100]:
def bandwidth_gbps(bytes_moved, latency_us):
    return bytes_moved / (latency_us * 1e-6) / 1e9

rows = []

for precision, problem in PROBLEMS.items():
    naive_latency_us = timings[precision]["Naive CUDA"]["latency_us"]
    logical_memory_bytes = problem["logical_memory_bytes"]

    for method in RUNNERS[precision]:
        latency_us = timings[precision][method]["latency_us"]
        effective_bandwidth = bandwidth_gbps(logical_memory_bytes, latency_us)
        rows.append(
            {
                "precision": precision,
                "method": method,
                "latency_us": latency_us,
                "bandwidth_GBps": effective_bandwidth,
                "percent_peak_bandwidth": effective_bandwidth / PEAK_BANDWIDTH_GBPS * 100.0,
                "speedup_vs_naive": naive_latency_us / latency_us,
                **correctness[(precision, method)],
            }
        )

results = pd.DataFrame(rows)
results


NameError: name 'timings' is not defined

## 12. Plots

Four figures compare FP16 and FP32 side-by-side for each method: latency, logical effective bandwidth, configured-peak utilization, and speedup over the naive kernel.

In [ ]:
import numpy as np

PLOT_SPECS = [
    ("latency_us", "GEMV latency", "Latency (microseconds)", "{:.1f}", "Lower is better"),
    ("bandwidth_GBps", "Logical GEMV memory bandwidth", "Bandwidth (GB/s)", "{:.1f}", "Higher is better"),
    (
        "percent_peak_bandwidth",
        "Configured peak memory bandwidth reached",
        "Percent of peak (%)",
        "{:.1f}%",
        "Higher is better",
    ),
    # ("speedup_vs_naive", "GEMV speedup vs naive CUDA", "Speedup (x)", "{:.2f}x", "Higher is better"),
]

PRECISION_COLORS = {
    "FP16": "#4C78A8",
    "FP32": "#F58518",
}

methods = results["method"].drop_duplicates().tolist()
precisions = results["precision"].drop_duplicates().tolist()

x = np.arange(len(methods))
bar_width = 0.8 / len(precisions)

for column, title, ylabel, value_format, direction_note in PLOT_SPECS:
    fig, ax = plt.subplots(figsize=(10, 5))

    for i, precision in enumerate(precisions):
        precision_results = (
            results[results["precision"] == precision]
            .set_index("method")
            .reindex(methods)
        )

        values = precision_results[column].to_numpy()
        offset = (i - (len(precisions) - 1) / 2) * bar_width

        bars = ax.bar(
            x + offset,
            values,
            width=bar_width,
            label=precision,
            color=PRECISION_COLORS.get(precision),
        )

        ax.bar_label(
            bars,
            labels=[value_format.format(value) for value in values],
            padding=3,
            fontsize=9,
        )

    ax.set_title(f"{title} ({direction_note})")
    ax.set_xlabel("Method")
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ymax = np.nanmax(results[column].to_numpy())
    ax.set_ylim(0, ymax * 1.15)
    ax.legend(
    title="Precision",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    borderaxespad=0,
    )

    fig.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show() 

NameError: name 'results' is not defined

## 13. Interpretation

GEMV is memory-bandwidth-bound because each weight is read once and reused very little.

For W[4096,4096]:

- FP16 weights read = 4096 × 4096 × 2 bytes ≈ 33.55 MB
- FP32 weights read = 4096 × 4096 × 4 bytes ≈ 67.11 MB
- x and y are small by comparison

The weight matrix dominates memory traffic, so the main optimization target is global memory bandwidth.

The naive CUDA kernel is slow because one thread does a full row dot product serially. The optimized variants expose more row-level parallelism:

- current optimized: one full block per row
- v1: one warp per row
- v2: one warp per row with tiled shared-vector staging
- v3: FP16 full-vector shared staging when the vector fits in default shared memory

The percentages below use `PEAK_BANDWIDTH_GBPS`, which is a configurable placeholder. Change it to the printed GPU's sustained or specified bandwidth before interpreting percent-of-peak. These are standalone kernel microbenchmark results; they do not by themselves establish an end-to-end LLM serving improvement.


In [102]:
for precision in results["precision"].unique():
    precision_results = results[results["precision"] == precision]
    best = precision_results.sort_values("latency_us", ascending=True).iloc[0]
    cublas = precision_results[precision_results["method"] == "cuBLAS direct"].iloc[0]
    print(
        f"{precision}: best method is {best['method']} at "
        f"{best['latency_us']:.2f} us and {best['bandwidth_GBps']:.2f} GB/s. "
        f"Speedup vs cuBLAS: {cublas['latency_us'] / best['latency_us']:.3f}x."
    )
